In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv(r"C:\Users\Allen\Documents\GitHub\thesis-research\combined_tweets_dataset\sample_v2\vibe_coding_combined_translated.csv")

In [3]:
df = df[["full_text_translated", "image_url"]]

In [4]:
df.head()

,full_text_translated,image_url
0,@karpathy me vibe coding and hope it can just ...,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg
1,vibes capital meets vibe coding = greatest eco...,NaN
2,vibe coding https://t.co/1hHVMmunrw,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg
3,can attest. vibe coding is hella fun and borde...,NaN
4,I have built a few app ideas in my spare time ...,NaN


In [5]:
import html
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Pastikan sudah download resource nltk
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')

stop_words = set(stopwords.words('english')) # Atau 'indonesian' jika data campur

def preprocess_tweet_v2(text):
    # 1. Decode HTML Entities (Penting untuk X/Twitter data)
    # Ini akan merubah &lt; menjadi < sehingga re.sub simbol bisa bekerja
    text = html.unescape(text)

    text = text.lower()

    # 2. Pola Regex kamu sudah bagus, pertahankan untuk URL, Mentions, Hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    # 3. Handle spesifik Vibe Coding sebelum hapus simbol
    text = re.sub(r'vibe\s+coding', 'vibecoding', text)
    text = re.sub(r'vibe\s+code', 'vibecode', text)
    text = re.sub(r'vibe\s+coded', 'vibecoded', text)

    # 4. Hapus simbol dan angka
    text = re.sub(r'[^\w\s]', ' ', text) # Ganti simbol dengan spasi agar kata tidak menempel
    text = re.sub(r'\d+', '', text)

    # 5. Tokenisasi & Stopwords (The missing piece)
    tokens = word_tokenize(text)
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    return " ".join(cleaned_tokens)

In [6]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet_v2)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [7]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

full_text_translated    19049
image_url                6000
dtype: int64

In [9]:
df_preprocessed.to_csv(r"./preprocessed_new_vibe_coding_tweets.csv", index=False)